# ML-09 — Validation and Research Claim Audit

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/SaaDasim05/Flyrank-ML-Internship/blob/main/work/notebooks/w06_validation_audit.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Two paper findings + my methodology questions

*Pick two findings from the FlyRank research paper. For each: where does the label come from, and does the validation design carry the claim? Constructive tone.*

### Finding 1 — Content age and growth/decline

**Paper finding:** The paper reports that growing content averaged about 184 days old and 3.2K words, while declining content averaged about 230 days old and 2.3K words. It describes this as an observational comparison and concludes that growing pages tend to be longer, younger, and slightly better positioned. :contentReference[oaicite:0]{index=0}

**Methodology question:** How exactly was the growing-versus-declining outcome defined, and were the age and word-count measurements taken from a time period independent of that outcome?

**Validation question:** Does the comparison support a broader recommendation about refreshing content, or is it descriptive evidence from this particular portfolio and reporting window?

I would treat the result as directional evidence rather than evidence that increasing word count or reducing age directly causes growth.

### Finding 2 — Age and freshness interact

**Paper finding:** The paper reports that old content that was recently refreshed can perform similarly to younger, recently refreshed content. In its age-freshness matrix, the “Young + Fresh” quadrant had a health score of 44.12, while “Old + Refreshed” was 44.62. The paper also identifies “Mid + Stale” as a decay zone and warns that the 365+ / long-unchanged cell is a small survivor-biased sample. :contentReference[oaicite:2]{index=2}

**Methodology question:** How were the age and freshness groups constructed, and how large are the cells supporting each comparison?

**Validation question:** Does the observed difference support a general refresh recommendation, or could selection effects and survivor bias explain part of the pattern, especially in the small 365+ × 361+ cell?

The paper itself warns that the small long-unchanged cell should not be used as headline evidence, so I would preserve that limitation in any interpretation.

## 2. My model under an honest split (before/after)

*Re-run your Week-5 model under a grouped or time-aware split. Show both numbers.*


I compare the same Random Forest, feature set, target, and Precision@50 metric under two validation designs.

The first is a random row split, which allows content items from the same client to appear in both training and testing. The second uses `GroupShuffleSplit` on `client_hash_id`, so all content from a client stays entirely within either training or testing.

The grouped split is the more appropriate primary validation design for this use case because the goal is to understand whether the ranking approach transfers to clients not seen during training.

In [3]:
import os
import duckdb
import pandas as pd
import numpy as np

from google.colab import userdata

from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split, GroupShuffleSplit

# -----------------------------
# Hugging Face / DuckDB setup
# -----------------------------

HF_TOKEN = userdata.get("HF_TOKEN")

if not HF_TOKEN:
    raise ValueError("HF_TOKEN not found in Colab Secrets.")

con = duckdb.connect()

con.execute(f"""
    CREATE OR REPLACE SECRET hf (
        TYPE HUGGINGFACE,
        TOKEN '{HF_TOKEN}'
    )
""")

REL = "hf://datasets/FlyRank/internship-warehouse"

TABLES = {
    "fact_daily":
        f"read_parquet('{REL}/fact_content_daily_performance/**/*.parquet')",
}

print("Warehouse connection ready.")

Warehouse connection ready.


In [5]:
# -----------------------------
# March = features
# April = future outcome
# -----------------------------

march = con.sql(f"""
    SELECT
        client_hash_id,
        content_hash_id,

        SUM(gsc_impressions) AS impressions_march,
        SUM(gsc_clicks) AS clicks_march,

        AVG(
            CASE
                WHEN gsc_data_available IS TRUE
                THEN gsc_avg_position
            END
        ) AS avg_position_march,

        SUM(sessions_organic) AS organic_sessions_march,

        COUNT(DISTINCT report_date) FILTER (
            WHERE gsc_impressions > 0
        ) AS days_with_impressions_march

    FROM {TABLES['fact_daily']}

    WHERE report_date >= DATE '2026-03-01'
      AND report_date < DATE '2026-04-01'

    GROUP BY 1, 2
""").df()

april = con.sql(f"""
    SELECT
        client_hash_id,
        content_hash_id,
        SUM(gsc_impressions) AS impressions_april

    FROM {TABLES['fact_daily']}

    WHERE report_date >= DATE '2026-04-01'
      AND report_date < DATE '2026-05-01'

    GROUP BY 1, 2
""").df()

data = march.merge(
    april,
    on=["client_hash_id", "content_hash_id"],
    how="inner"
)

data["is_declining"] = (
    data["impressions_april"] <
    0.8 * data["impressions_march"]
).astype(int)

print(f"Rows: {len(data):,}")
print(f"Clients: {data['client_hash_id'].nunique():,}")
print(f"Decline rate: {data['is_declining'].mean():.3f}")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Rows: 331,436
Clients: 55
Decline rate: 0.284


In [6]:
content_dates = con.sql(f"""
    SELECT
        client_hash_id,
        content_hash_id,
        content_updated_date,
        word_count
    FROM read_parquet('{REL}/dim_content.parquet')
""").df()

content_dates["content_updated_date"] = pd.to_datetime(
    content_dates["content_updated_date"],
    errors="coerce"
)

decision_date = pd.Timestamp("2026-03-31")

content_dates["days_since_update"] = (
    decision_date - content_dates["content_updated_date"]
).dt.days.clip(lower=0)

data = data.merge(
    content_dates[
        [
            "client_hash_id",
            "content_hash_id",
            "days_since_update",
            "word_count"
        ]
    ],
    on=["client_hash_id", "content_hash_id"],
    how="left"
)

print(f"Rows after metadata join: {len(data):,}")
print(f"Missing update dates: {data['days_since_update'].isna().sum():,}")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Rows after metadata join: 331,436
Missing update dates: 0


In [9]:
# Build the comparison table using the already-trained models

random_test_clients = model_df.loc[
    X_test.index, "client_hash_id"
].nunique()

group_test_clients = model_df.iloc[
    test_idx
]["client_hash_id"].nunique()

comparison = pd.DataFrame({
    "validation": [
        "Random row split",
        "Client GroupShuffleSplit"
    ],
    "Precision@50": [
        random_p50,
        group_p50
    ],
    "test_rows": [
        len(y_test),
        len(y_group_test)
    ],
    "test_clients": [
        random_test_clients,
        group_test_clients
    ]
})

comparison

,validation,Precision@50,test_rows,test_clients
0,Random row split,0.80,25340,37
1,Client GroupShuffleSplit,0.68,20768,10


### Before vs after

Under the random row split, the Random Forest achieved a measured Precision@50 of **0.80**. Under the stricter client-level GroupShuffleSplit, Precision@50 was **0.68**, with 10 held-out clients.

The lower score under client-level validation suggests that the random split may give a more optimistic estimate because content items from the same clients can appear in both training and testing. The grouped result is the more relevant evidence for whether the model transfers to unseen clients.

This comparison is directional and comes from one split; it should not be interpreted as a universal estimate of future performance.

## 3. Leakage audit

*The same hunt from Week 3, on your final feature set.*

In [10]:
forbidden_features = {
    "impressions_april",
    "is_declining",
    "trend_pct",
    "impressions_future",
    "future_impressions"
}

leaked = forbidden_features.intersection(model_features)

print("Model features:")
print(model_features)

print("\nPotential forbidden features found:")
print(leaked)

print("\nLeakage audit passed:", len(leaked) == 0)

Model features:
['impressions_march', 'clicks_march', 'avg_position_march', 'organic_sessions_march', 'days_with_impressions_march', 'days_since_update', 'word_count']

Potential forbidden features found:
set()

Leakage audit passed: True


In [11]:
feature_timing = pd.DataFrame({
    "feature": model_features,
    "available_by_2026_03_31": [
        True,
        True,
        True,
        True,
        True,
        True,
        True
    ]
})

feature_timing

,feature,available_by_2026_03_31
0,impressions_march,True
1,clicks_march,True
2,avg_position_march,True
3,organic_sessions_march,True
4,days_with_impressions_march,True
5,days_since_update,True
6,word_count,True


## 3. Leakage audit

The model features are restricted to information available by the March 31, 2026 decision point. The April outcome (`impressions_april`) and the target (`is_declining`) are excluded from the feature set.

I also excluded `trend_pct`, because a trend field derived from the outcome could leak information that would not be available when making the prediction.

The leakage audit found no forbidden future or target-derived fields in the final model feature list.

The validation design also prevents client overlap between training and testing in the grouped evaluation.

In [12]:
group_errors = model_df.iloc[test_idx].copy()

group_errors["model_score"] = group_scores

group_errors["prediction"] = (
    group_errors["model_score"] >= 0.5
).astype(int)

false_positives = group_errors[
    (group_errors["prediction"] == 1) &
    (group_errors["is_declining"] == 0)
].sort_values("model_score", ascending=False)

false_negatives = group_errors[
    (group_errors["prediction"] == 0) &
    (group_errors["is_declining"] == 1)
].sort_values("model_score", ascending=False)

print("False positives:", len(false_positives))
print("False negatives:", len(false_negatives))

print("\nFalse-positive examples:")
display(
    false_positives[
        [
            "client_hash_id",
            "content_hash_id",
            "model_score",
            "is_declining",
            "impressions_march",
            "avg_position_march",
            "days_since_update",
            "word_count"
        ]
    ].head(10)
)

print("\nFalse-negative examples:")
display(
    false_negatives[
        [
            "client_hash_id",
            "content_hash_id",
            "model_score",
            "is_declining",
            "impressions_march",
            "avg_position_march",
            "days_since_update",
            "word_count"
        ]
    ].head(10)
)

False positives: 3843
False negatives: 4442

False-positive examples:


,client_hash_id,content_hash_id,model_score,is_declining,impressions_march,avg_position_march,days_since_update,word_count
10006,client_c182d11e4862a37d,content_5bb547576ac2555c,0.990000,0,5.0,3.600000,0,1446
173946,client_65de48885f4ef01b,content_192d903c68674e57,0.990000,0,3.0,4.333333,0,1328
174499,client_65de48885f4ef01b,content_3e77b18cafcf5385,0.986667,0,2.0,4.500000,0,1352
38295,client_c182d11e4862a37d,content_014aac56410f2cb2,0.986667,0,1.0,9.000000,0,1014
54155,client_65de48885f4ef01b,content_1bd58570e2cbccfc,0.980000,0,2.0,4.500000,0,1132
291233,client_fef1a8f436438636,content_96a22d5cc2029644,0.977512,0,1.0,0.000000,0,1482
198468,client_1a730cb2640a1abf,content_3391671dc1eecda4,0.976667,0,1.0,8.000000,0,2532
316658,client_def0955f7a377868,content_e74c83a3014dcac1,0.970000,0,4.0,7.166667,0,1688
126095,client_fef1a8f436438636,content_c040c76976760cb3,0.970000,0,627.0,4.888770,0,1499
8487,client_65de48885f4ef01b,content_4741a4331c0e2c53,0.966667,0,1.0,7.000000,34,1491



False-negative examples:


,client_hash_id,content_hash_id,model_score,is_declining,impressions_march,avg_position_march,days_since_update,word_count
162546,client_b77d0d5f08f05e64,content_58a5c5d5e03da3f5,0.496667,1,13.0,33.962963,0,2368
160060,client_b77d0d5f08f05e64,content_5d7f0731f107ffe7,0.496667,1,12.0,15.786667,0,2768
7775,client_65de48885f4ef01b,content_20cf73e4c71fe34f,0.496667,1,90.0,33.063004,34,1550
125696,client_fef1a8f436438636,content_ad9a788fc9b58968,0.496667,1,41.0,7.768182,0,2423
311317,client_fef1a8f436438636,content_08c97bbc72bac9aa,0.496667,1,316.0,8.578708,0,2665
125426,client_fef1a8f436438636,content_b76ca098d5b11450,0.496667,1,2413.0,10.704036,0,3150
20157,client_fef1a8f436438636,content_35b8769ea4dc6429,0.496667,1,1205.0,8.737111,0,2855
146356,client_fef1a8f436438636,content_8c16b8a0e310a05b,0.496667,1,870.0,4.687695,0,2705
146412,client_fef1a8f436438636,content_f52d3867f8a7be77,0.496667,1,877.0,5.958768,0,1720
146775,client_fef1a8f436438636,content_4f906596de95c85c,0.496667,1,2625.0,37.679908,0,1447


## 3. Leakage audit

The model uses only features available by the March 31, 2026 decision point. The April outcome and the target are excluded from the predictors.

The final feature set contains:

- `impressions_march`
- `clicks_march`
- `avg_position_march`
- `organic_sessions_march`
- `days_with_impressions_march`
- `days_since_update`
- `word_count`

I deliberately excluded future/outcome-derived fields such as `impressions_april`, `is_declining`, and `trend_pct`.

The audit found no forbidden target or future variables in the final feature list. The client-group validation also prevents the same client from appearing in both training and testing.

## 4. Claim rewrite

*Take your own boldest sentence and rewrite it in safe language: observed, measured, directional, decision-support.*

### Original claim

The Random Forest predicts which pages need refreshing better than the Week-4 baseline.

### Revised claim

In the evaluated experiment, the Random Forest produced measurable ranking performance under both random and client-group validation. Precision@50 was 0.80 under the random row split and 0.68 under the stricter client-group split.

The lower grouped result suggests that the random split may give a more optimistic estimate when content items from the same clients can appear in both training and testing. I therefore treat the 0.68 client-group result as the more relevant directional evidence for transfer to unseen clients.

The model produced 3,843 false positives and 4,442 false negatives under the grouped evaluation. Some false negatives had predicted scores close to 0.50, showing that borderline cases remain difficult to separate cleanly.

I therefore cannot claim that the model identifies pages that definitely need refreshing or that refreshing a high-scoring page will improve search performance. The evidence supports describing the model as a decision-support ranking signal for human review.

## Self-check
- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors
- [x] No client names, URLs, or private queries
- [x] Claims use observed, measured, directional, decision-support language
- [x] Committed to my repo under work/notebooks/